In [13]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile, f_classif
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
import joblib
from pathlib import Path

In [14]:
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-dnam-40k")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.dnam.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

base_dir = Path.cwd()
features_path = base_dir / "output" / "selected_features.4.txt"


output_dir = "/home/mariya/Data/pallas/models/dnam"
random_seed = 42
n_jobs = 4
model_path = os.path.join(output_dir, "rf_model.pkl")
hyperparams_path = os.path.join(output_dir, "hyperparams.pkl")

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")
print(f"model_path: {model_path}")
print(f"hyperparams_path: {hyperparams_path}")

# Hardcoded variables
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-dnam-40k/features.dnam.csv
y_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-dnam-40k/labels.csv
output_dir: /home/mariya/Data/pallas/models/dnam
random_seed: 42
n_jobs: 4
model_path: /home/mariya/Data/pallas/models/dnam/rf_model.pkl
hyperparams_path: /home/mariya/Data/pallas/models/dnam/hyperparams.pkl


In [15]:
# Read in selected features
with open(features_path, "r") as f:
    selected_features = [line.strip() for line in f.readlines()]
print(f"Number of selected features: {len(selected_features)}")

Number of selected features: 1227


In [16]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)

# Subset X_train to selected features
X_train = X_train[selected_features]


In [17]:
y_train = pd.read_csv(y_train_path, index_col=0)
print("y_train class distribution:")
print(y_train.value_counts())

# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

y_train class distribution:
class                   
AML, KMT2A-r                241
B-ALL, high hyperdiploid    229
B-ALL, ETV6::RUNX1          199
AML, RUNX1::RUNX1T1         174
AML, NPM1alt                130
AML, CBFB::MYH11            127
Non-leukemic                 95
T-ALL                        73
AML, NUP98-r                 69
AML, CEBPAalt                55
B-ALL, PAX5alt               47
B-ALL, KMT2A-r               43
B-ALL, TCF3::PBX1            26
B-ALL, DUX4-r                22
AML, CBFA2T3::GLIS2          22
AML, DEK::NUP214             19
B-ALL, BCR::ABL1             17
AML, APL-PML::RARA           16
AML, ETV6-r                  16
B-ALL, ETV6::RUNX1-like      15
B-ALL, iAMP21                15
B-ALL, ZNF384-r              15
B-ALL, BCR::ABL1-like        14
AML, KAT6A-r                  8
AML, PICALM::MLLT10           6
AML, FUS::ERG                 6
B-ALL, MEF2D-r                6
B-ALL, PAX5 P80R              6
Name: count, dtype: int64


np.int64(6)

In [18]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_train

array(['B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', ..., 'AML, APL-PML::RARA',
       'AML, APL-PML::RARA', 'AML, APL-PML::RARA'],
      shape=(1711,), dtype=object)

In [19]:
# Reduce memory usage by converting to float32
X_train = X_train.astype(np.float32)

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV

N_ITER = 30
N_REPEATS = 1
N_FOLDS_INITIAL = max(5, min(min_class_count, 10))

rf = RandomForestClassifier(
    random_state=random_seed,
    n_jobs=1           # parallelism handled at search level
)

param_grid = {
    'n_estimators': [50], # Set to fewer estimators for faster search; can be increased for final model
    'max_depth': [20, 30, 40],
    # min_samples_leaf — hard upper bound of 2 given min class = 5
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 0.15, 0.25, 0.3],
    'class_weight': ['balanced'],
}

cv = RepeatedStratifiedKFold(
    n_splits=N_FOLDS_INITIAL,
    n_repeats=N_REPEATS,
    random_state=random_seed
)

search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,             # parallelise at search level
    random_state=random_seed,
    verbose=2,
    refit=True
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (f1_macro):",   search.best_score_)

best_model = search.best_estimator_

Fitting 6 folds for each of 24 candidates, totalling 144 fits


/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 24 is smaller than n_iter=30. Running 24 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END class_weight=balanced, max_depth=20, max_features=sqrt, min_samples_leaf=1, n_estimators=50; total time=   6.4s
[CV] END class_weight=balanced, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=50; total time=   7.0s
[CV] END class_weight=balanced, max_depth=20, max_features=sqrt, min_samples_leaf=1, n_estimators=50; total time=   8.2s
[CV] END class_weight=balanced, max_depth=20, max_features=sqrt, min_samples_leaf=1, n_estimators=50; total time=   7.6s
[CV] END class_weight=balanced, max_depth=20, max_features=sqrt, min_samples_leaf=1, n_estimators=50; total time=   7.5s
[CV] END class_weight=balanced, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=50; total time=   8.5s
[CV] END class_weight=balanced, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=50; total time=   6.9s
[CV] END class_weight=balanced, max_depth=20, max_features=sqrt, min_samples_leaf=2, n_estimators=50; total time=  11.0s
[CV] END class_weight=balanced, 

In [21]:

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparams_path)

# Save the model
joblib.dump(best_model, model_path)

['/home/mariya/Data/pallas/models/dnam/rf_model.pkl']

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
import joblib, os

N_REPEATS = 10

# Load and parse best params
with open(hyperparams_path, "rb") as f:
    best_params = joblib.load(f)

# rf_params = {
#     # k.replace('rf__', ''): v
#     for k, v in best_params.items()
#     # if k.startswith('rf__')
# }
rf_params = best_params
rf_params['n_estimators'] = FINAL_N_ESTIMATORS

print("Final RF parameters:", rf_params)

final_rf = RandomForestClassifier(
    **rf_params,
    n_jobs=1,
    random_state=random_seed
)

# Cross-validate with train score to monitor gap
# n_splits = max(5, min(min_class_count, 10))
# cv = RepeatedStratifiedKFold(
#     n_splits=n_splits,
#     n_repeats=N_REPEATS,
#     random_state=random_seed
# )

# cv_results = cross_validate(
#     final_rf,
#     X_train,
#     y_train,
#     cv=cv,
#     scoring="f1_macro",
#     n_jobs=-1,
#     verbose=2,
#     return_train_score=True
# )

# train_mean = cv_results['train_score'].mean()
# train_std  = cv_results['train_score'].std()
# val_mean   = cv_results['test_score'].mean()
# val_std    = cv_results['test_score'].std()

# print(f"Train F1-macro: {train_mean:.3f} ± {train_std:.3f}")
# print(f"Val   F1-macro: {val_mean:.3f} ± {val_std:.3f}")
# print(f"Gap:            {train_mean - val_mean:.3f}")

# Fit on all training data
final_rf.fit(X_train, y_train)


Final RF parameters: {'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 20, 'class_weight': 'balanced'}


,n_estimators,500
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,0.3
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:

# ----------------------
# Save everything
# ----------------------

# Save search best params separately for reference
joblib.dump(best_params, hyperparams_path.replace('.pkl', '_search.pkl'))

# Save final hyperparameters — store rf__ prefixed params plus fs percentile
# so the format matches what was saved from the search step
final_params = {f'rf__{k}': v for k, v in rf_params.items()}
joblib.dump(final_params, hyperparams_path)

# Save the model
joblib.dump(final_rf, model_path)

# Save model separately as well for easy loading without the feature selection step if needed
model_path = os.path.join(output_dir, "rf_model.pkl")
joblib.dump(final_rf, model_path)

print(f"Saved hyperparameters to {hyperparams_path}")
print(f"Saved final RF model to {model_path}")

Saved hyperparameters to /home/mariya/Data/pallas/models/dnam/hyperparams.pkl
Saved final RF model to /home/mariya/Data/pallas/models/dnam/rf_model.pkl
